# 🧠 Lab: Building a GAN from Scratch with PyTorch

**Course:** Deep Learning Lab  
**Topic:** Generative Adversarial Networks (GANs)  
**Estimated Time:** 2–3 hours

---

## 📌 Lab Objectives

By the end of this lab, you will be able to:
1. Understand the architecture and training dynamics of GANs
2. Use key PyTorch layers: `Conv2d`, `ConvTranspose2d`, `BatchNorm2d`, and activation functions
3. Build a **Discriminator** and a **Generator** from scratch
4. Implement the **two-stage GAN training loop**
5. Visualize generated images and training losses

---

## 🧩 Background: How GANs Work

A GAN consists of two neural networks locked in a **minimax game**:

| Component | Role | Input | Output |
|---|---|---|---|
| **Generator (G)** | Creates fake images | Random noise vector `z` | Fake image |
| **Discriminator (D)** | Classifies real vs. fake | An image | Probability [0, 1] |

The objective:
$$\min_G \max_D \; \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

**Intuition:**
- The **Discriminator** tries to correctly label real images as 1 and fake images as 0
- The **Generator** tries to fool the Discriminator into labeling its outputs as 1
- They improve together until G produces indistinguishable images

### Two-Stage Training Process

```
Each training iteration:
┌─────────────────────────────────────────────────────────┐
│  STAGE 1 — Train Discriminator                          │
│  • Feed REAL images → D should output 1                 │
│  • Feed FAKE images (G(z)) → D should output 0          │
│  • Compute loss, backprop ONLY through D                │
│  • Update D weights (G weights are FROZEN)              │
├─────────────────────────────────────────────────────────┤
│  STAGE 2 — Train Generator                              │
│  • Feed new FAKE images through D                       │
│  • Compute loss: G wants D to output 1 for fake images  │
│  • Backprop through D (but update ONLY G weights)       │
│  • D weights are FROZEN during this stage               │
└─────────────────────────────────────────────────────────┘
```

---

## 🔧 PyTorch Layer Refresher

### `nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)`
- Used in the **Discriminator** to downsample feature maps
- Learns spatial features by sliding a kernel across the input
- Output size: `floor((H + 2P - K) / S) + 1`

### `nn.ConvTranspose2d(in_channels, out_channels, kernel_size, stride, padding)`
- Used in the **Generator** to upsample ("deconvolve") feature maps
- Learns to expand a small latent vector into a full image
- Output size: `(H-1)*S - 2P + K`

### `nn.BatchNorm2d(num_features)`
- Normalizes activations across the batch dimension
- Stabilizes GAN training significantly

### Activation Functions
| Function | Formula | Used In |
|---|---|---|
| `nn.ReLU()` | max(0, x) | Basic layers |
| `nn.LeakyReLU(0.2)` | max(0.2x, x) | Discriminator (avoids dead neurons) |
| `nn.Tanh()` | (e^x - e^-x)/(e^x + e^-x) | Generator output (range [-1, 1]) |
| `nn.Sigmoid()` | 1/(1+e^-x) | Discriminator output (range [0, 1]) |


---
## Part 0: Setup and Imports

In [ ]:
# Install dependencies if needed
# !pip install torch torchvision matplotlib numpy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import os

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

---
## Part 1: Hyperparameters and Data Loading

We will train on the **MNIST** dataset (handwritten digits). Each image is 28×28 grayscale.  
We resize to 32×32 to make the architecture cleaner with powers of 2.

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
IMAGE_SIZE   = 32        # Resize MNIST to 32×32
NC           = 1         # Number of channels (grayscale = 1, RGB = 3)
NZ           = 100       # Size of the latent noise vector z
NGF          = 64        # Feature map size inside Generator
NDF          = 64        # Feature map size inside Discriminator
BATCH_SIZE   = 128
NUM_EPOCHS   = 20
LR           = 0.0002    # Learning rate (DCGAN paper recommendation)
BETA1        = 0.5       # Adam beta1 (DCGAN paper recommendation)

# ── Data Pipeline ────────────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))   # Normalize to [-1, 1] to match Tanh output
])

dataset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)

dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)

print(f"Dataset size : {len(dataset):,} images")
print(f"Batches/epoch: {len(dataloader)}")
print(f"Image shape  : {dataset[0][0].shape}")

# Visualise a batch of real images
real_batch = next(iter(dataloader))
plt.figure(figsize=(10, 4))
plt.axis("off")
plt.title("Real Training Images (MNIST)", fontsize=14)
plt.imshow(
    np.transpose(
        vutils.make_grid(real_batch[0][:64], padding=2, normalize=True).numpy(),
        (1, 2, 0)
    ),
    cmap="gray"
)
plt.show()

---
## Part 2: Weight Initialisation

The DCGAN paper recommends initialising all conv/batchnorm weights from  
a **Normal distribution N(0, 0.02)**. This is a critical trick for training stability.

In [ ]:
def weights_init(m):
    """
    Custom weight initialiser as per the DCGAN paper.
    Applied to both Generator and Discriminator after construction.
    """
    classname = m.__class__.__name__
    if "Conv" in classname:                        # Conv2d and ConvTranspose2d
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)  # BN gamma ~ N(1, 0.02)
        nn.init.constant_(m.bias.data, 0)           # BN beta  = 0

print("Weight initialiser defined.")

---
## Part 3: Build the Generator

### Architecture Overview

The Generator maps a **latent noise vector z** (shape: `[B, NZ, 1, 1]`) to a fake image (shape: `[B, NC, 32, 32]`) using a stack of **transposed convolutions** (also called fractionally strided convolutions).

```
z [B, 100, 1, 1]
    │ ConvTranspose2d(100 → 256, k=4, s=1, p=0)  → [B, 256, 4, 4]
    │ BatchNorm + ReLU
    │ ConvTranspose2d(256 → 128, k=4, s=2, p=1)  → [B, 128, 8, 8]
    │ BatchNorm + ReLU
    │ ConvTranspose2d(128 →  64, k=4, s=2, p=1)  → [B,  64, 16, 16]
    │ BatchNorm + ReLU
    │ ConvTranspose2d( 64 →   1, k=4, s=2, p=1)  → [B,   1, 32, 32]
    │ Tanh → outputs in range [-1, 1]
```

### ✏️ Exercise 3.1
**Complete the Generator class below.** The architecture is described above.  
Use `nn.Sequential` to stack layers inside `self.main`.

> **Hints:**
> - Use `ConvTranspose2d` for upsampling
> - Add `BatchNorm2d` after every conv layer except the last
> - Use `ReLU(inplace=True)` as activations (except the last layer)
> - Final activation must be `Tanh` (to match data normalisation range)

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────

class Generator(nn.Module):
    def __init__(self, nz, ngf, nc):
        """
        Args:
            nz  (int): Size of latent noise vector
            ngf (int): Base feature map size
            nc  (int): Number of output channels (1 for grayscale)
        """
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            # TODO: Block 1 — nz → ngf*4, spatial: 1×1 → 4×4
            # (ConvTranspose2d, BatchNorm2d, ReLU)

            # TODO: Block 2 — ngf*4 → ngf*2, spatial: 4×4 → 8×8

            # TODO: Block 3 — ngf*2 → ngf, spatial: 8×8 → 16×16

            # TODO: Block 4 — ngf → nc, spatial: 16×16 → 32×32
            # (ConvTranspose2d, Tanh) — NO BatchNorm on output layer
        )

    def forward(self, x):
        return self.main(x)


# ─────────────────────────────────────────────────────────────────────────────

### ✅ Solution: Generator

In [ ]:
# ── SOLUTION ──────────────────────────────────────────────────────────────────

class Generator(nn.Module):
    def __init__(self, nz, ngf, nc):
        super(Generator, self).__init__()
        self.main = nn.Sequential(

            # Block 1: Input z of shape [B, nz, 1, 1] → [B, ngf*4, 4, 4]
            # stride=1, padding=0: output = (1-1)*1 - 2*0 + 4 = 4  ✓
            nn.ConvTranspose2d(nz, ngf * 4, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(inplace=True),

            # Block 2: [B, ngf*4, 4, 4] → [B, ngf*2, 8, 8]
            # stride=2, padding=1: output = (4-1)*2 - 2*1 + 4 = 8  ✓
            nn.ConvTranspose2d(ngf * 4, ngf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(inplace=True),

            # Block 3: [B, ngf*2, 8, 8] → [B, ngf, 16, 16]
            # stride=2, padding=1: output = (8-1)*2 - 2*1 + 4 = 16  ✓
            nn.ConvTranspose2d(ngf * 2, ngf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(inplace=True),

            # Block 4 (Output): [B, ngf, 16, 16] → [B, nc, 32, 32]
            # stride=2, padding=1: output = (16-1)*2 - 2*1 + 4 = 32  ✓
            # NO BatchNorm on the output layer
            nn.ConvTranspose2d(ngf, nc, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()   # Output range: [-1, 1] — matches data normalisation
        )

    def forward(self, x):
        return self.main(x)


# Instantiate and apply weight init
netG = Generator(nz=NZ, ngf=NGF, nc=NC).to(device)
netG.apply(weights_init)

print("Generator Architecture:")
print("=" * 60)
print(netG)
print("=" * 60)

# Quick sanity check
test_noise = torch.randn(4, NZ, 1, 1).to(device)
test_out   = netG(test_noise)
print(f"\nInput shape : {test_noise.shape}")
print(f"Output shape: {test_out.shape}")
assert test_out.shape == (4, NC, IMAGE_SIZE, IMAGE_SIZE), "Shape mismatch!"
print("✅ Generator output shape is correct!")
print(f"Output range: [{test_out.min():.3f}, {test_out.max():.3f}] (should be within [-1, 1])")

---
## Part 4: Build the Discriminator

### Architecture Overview

The Discriminator is essentially a **CNN binary classifier** that maps an image (real or fake) to a single probability score.

```
Image [B, 1, 32, 32]
    │ Conv2d( 1 →  64, k=4, s=2, p=1) → [B, 64, 16, 16]
    │ LeakyReLU(0.2)
    │ Conv2d(64 → 128, k=4, s=2, p=1) → [B, 128, 8, 8]
    │ BatchNorm + LeakyReLU(0.2)
    │ Conv2d(128→ 256, k=4, s=2, p=1) → [B, 256, 4, 4]
    │ BatchNorm + LeakyReLU(0.2)
    │ Conv2d(256→   1, k=4, s=1, p=0) → [B, 1, 1, 1]
    │ Sigmoid → probability in [0, 1]
```

> **Why LeakyReLU in the Discriminator?**  
> Regular ReLU zeros out negative activations, which can lead to "dead neurons" in the discriminator. LeakyReLU with a small slope (0.2) preserves gradient flow for negative values.

### ✏️ Exercise 4.1
**Complete the Discriminator class below.**

> **Hints:**
> - Use `Conv2d` for downsampling (opposite of Generator)
> - NO BatchNorm on the first layer
> - Use `LeakyReLU(0.2, inplace=True)` as activations
> - Final activation is `Sigmoid`

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────

class Discriminator(nn.Module):
    def __init__(self, nc, ndf):
        """
        Args:
            nc  (int): Number of input image channels
            ndf (int): Base feature map size
        """
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            # TODO: Block 1 — nc → ndf, spatial: 32×32 → 16×16
            # NO BatchNorm on first layer!

            # TODO: Block 2 — ndf → ndf*2, spatial: 16×16 → 8×8

            # TODO: Block 3 — ndf*2 → ndf*4, spatial: 8×8 → 4×4

            # TODO: Block 4 — ndf*4 → 1, spatial: 4×4 → 1×1
            # (Conv2d, Sigmoid) — single probability score
        )

    def forward(self, x):
        return self.main(x)


# ─────────────────────────────────────────────────────────────────────────────

### ✅ Solution: Discriminator

In [ ]:
# ── SOLUTION ──────────────────────────────────────────────────────────────────

class Discriminator(nn.Module):
    def __init__(self, nc, ndf):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(

            # Block 1 (Input): [B, nc, 32, 32] → [B, ndf, 16, 16]
            # output = floor((32 + 2*1 - 4) / 2) + 1 = 16  ✓
            # NO BatchNorm on first discriminator layer
            nn.Conv2d(nc, ndf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # Block 2: [B, ndf, 16, 16] → [B, ndf*2, 8, 8]
            nn.Conv2d(ndf, ndf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # Block 3: [B, ndf*2, 8, 8] → [B, ndf*4, 4, 4]
            nn.Conv2d(ndf * 2, ndf * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # Block 4 (Output): [B, ndf*4, 4, 4] → [B, 1, 1, 1]
            # stride=1, padding=0: output = floor((4+0-4)/1)+1 = 1  ✓
            nn.Conv2d(ndf * 4, 1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid()   # Output: probability in [0, 1]
        )

    def forward(self, x):
        return self.main(x)


# Instantiate and apply weight init
netD = Discriminator(nc=NC, ndf=NDF).to(device)
netD.apply(weights_init)

print("Discriminator Architecture:")
print("=" * 60)
print(netD)
print("=" * 60)

# Sanity check
test_img = torch.randn(4, NC, IMAGE_SIZE, IMAGE_SIZE).to(device)
test_pred = netD(test_img)
print(f"\nInput shape : {test_img.shape}")
print(f"Output shape: {test_pred.shape}")
assert test_pred.shape == (4, 1, 1, 1), "Shape mismatch!"
print("✅ Discriminator output shape is correct!")
print(f"Output range: [{test_pred.min():.3f}, {test_pred.max():.3f}] (should be within [0, 1])")

# Parameter count
G_params = sum(p.numel() for p in netG.parameters())
D_params = sum(p.numel() for p in netD.parameters())
print(f"\nGenerator parameters    : {G_params:,}")
print(f"Discriminator parameters: {D_params:,}")

---
## Part 5: Loss Function and Optimisers

### Loss Function

GANs use **Binary Cross-Entropy (BCE) loss**:
$$\mathcal{L}_{BCE} = -[y \cdot \log(\hat{y}) + (1-y) \cdot \log(1-\hat{y})]$$

- For **real images**: target label `y = 1` → D wants to output values close to 1
- For **fake images** (training D): target label `y = 0` → D wants to output values close to 0  
- For **fake images** (training G): target label `y = 1` → G wants D to think fakes are real

### ✏️ Exercise 5.1
Define the loss function and separate optimisers for G and D.

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────

# 1. Define BCE loss
criterion = None  # TODO

# 2. Optimiser for Discriminator (Adam, lr=LR, betas=(BETA1, 0.999))
optimizerD = None  # TODO

# 3. Optimiser for Generator
optimizerG = None  # TODO

# ─────────────────────────────────────────────────────────────────────────────

### ✅ Solution: Loss and Optimisers

In [ ]:
# ── SOLUTION ──────────────────────────────────────────────────────────────────

# Binary Cross-Entropy loss
criterion = nn.BCELoss()

# IMPORTANT: Separate optimisers for each network!
# This lets us update only D in Stage 1 and only G in Stage 2
optimizerD = optim.Adam(netD.parameters(), lr=LR, betas=(BETA1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=LR, betas=(BETA1, 0.999))

# Fixed noise for visualisation — same noise vector across epochs to track progress
fixed_noise = torch.randn(64, NZ, 1, 1, device=device)

# Label conventions
REAL_LABEL = 1.0
FAKE_LABEL = 0.0

print("Loss function : Binary Cross-Entropy (BCELoss)")
print(f"Optimizer D   : Adam(lr={LR}, betas=({BETA1}, 0.999))")
print(f"Optimizer G   : Adam(lr={LR}, betas=({BETA1}, 0.999))")
print("✅ Ready to train!")

---
## Part 6: The Two-Stage Training Loop

This is the heart of GAN training. Study the pseudocode carefully before implementing:

```
for each batch of real images:

  ╔══════════════════════════════════════════════════════╗
  ║  STAGE 1: Update Discriminator                       ║
  ║                                                      ║
  ║  # 1a. Train on REAL images                          ║
  ║  real_imgs → D → D(x)                                ║
  ║  loss_D_real = BCE(D(x), labels_of_1s)               ║
  ║                                                      ║
  ║  # 1b. Train on FAKE images                          ║
  ║  z = sample_noise()                                  ║
  ║  fake_imgs = G(z)              # G is frozen here    ║
  ║  fake_imgs → D → D(G(z))                             ║
  ║  loss_D_fake = BCE(D(G(z)), labels_of_0s)            ║
  ║                                                      ║
  ║  loss_D = loss_D_real + loss_D_fake                  ║
  ║  optimizerD.zero_grad()                              ║
  ║  loss_D.backward()                                   ║
  ║  optimizerD.step()   ← UPDATE D ONLY                 ║
  ╚══════════════════════════════════════════════════════╝

  ╔══════════════════════════════════════════════════════╗
  ║  STAGE 2: Update Generator                           ║
  ║                                                      ║
  ║  z = sample_noise()                                  ║
  ║  fake_imgs = G(z)                                    ║
  ║  fake_imgs → D → D(G(z))    # D is frozen here      ║
  ║  # G WANTS D to output 1 for fakes → use label=1    ║
  ║  loss_G = BCE(D(G(z)), labels_of_1s)                 ║
  ║                                                      ║
  ║  optimizerG.zero_grad()                              ║
  ║  loss_G.backward()                                   ║
  ║  optimizerG.step()   ← UPDATE G ONLY                 ║
  ╚══════════════════════════════════════════════════════╝
```

### ✏️ Exercise 6.1
Implement the full two-stage training loop below.

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# Implement the training loop following the pseudocode above

G_losses, D_losses = [], []
img_list = []
iters = 0

for epoch in range(NUM_EPOCHS):
    for i, (real_imgs, _) in enumerate(dataloader):

        real_imgs = real_imgs.to(device)
        b_size = real_imgs.size(0)

        # ── STAGE 1: Train Discriminator ──────────────────────────────────────
        # TODO

        # ── STAGE 2: Train Generator ──────────────────────────────────────────
        # TODO

        iters += 1

# ─────────────────────────────────────────────────────────────────────────────

### ✅ Solution: Full Training Loop

In [ ]:
# ── SOLUTION ──────────────────────────────────────────────────────────────────

G_losses, D_losses = [], []
img_list = []
iters = 0

print("Starting Training...")
print(f"{'Epoch':>6} {'Batch':>6} {'Loss_D':>10} {'Loss_G':>10} {'D(x)':>8} {'D(G(z))':>10}")
print("-" * 58)

for epoch in range(NUM_EPOCHS):
    for i, (real_imgs, _) in enumerate(dataloader):

        real_imgs = real_imgs.to(device)
        b_size    = real_imgs.size(0)

        # ─────────────────────────────────────────────────────────────────────
        # STAGE 1: Update Discriminator
        # Maximise: log D(x) + log(1 - D(G(z)))
        # ─────────────────────────────────────────────────────────────────────
        netD.zero_grad()

        # 1a. REAL images — D should output 1
        label = torch.full((b_size,), REAL_LABEL, dtype=torch.float, device=device)
        output  = netD(real_imgs).view(-1)          # Flatten [B,1,1,1] → [B]
        lossD_real = criterion(output, label)
        lossD_real.backward()                        # Accumulate gradients
        D_x = output.mean().item()                   # Avg D score on real images

        # 1b. FAKE images — D should output 0
        noise = torch.randn(b_size, NZ, 1, 1, device=device)
        fake  = netG(noise)                          # Generate fakes (G not updated)
        label.fill_(FAKE_LABEL)
        # Use .detach() to avoid computing gradients through G during D's update!
        output    = netD(fake.detach()).view(-1)
        lossD_fake = criterion(output, label)
        lossD_fake.backward()                        # Accumulate gradients
        D_G_z1 = output.mean().item()               # Avg D score on fakes (before G update)

        # Total D loss and parameter update
        lossD = lossD_real + lossD_fake
        optimizerD.step()                            # Update ONLY D

        # ─────────────────────────────────────────────────────────────────────
        # STAGE 2: Update Generator
        # Maximise: log D(G(z))  [Non-saturating version of the original objective]
        # ─────────────────────────────────────────────────────────────────────
        netG.zero_grad()

        label.fill_(REAL_LABEL)                      # G wants D to think fakes are REAL
        # Forward through D again (no detach — we need grads to flow back to G)
        output = netD(fake).view(-1)
        lossG  = criterion(output, label)
        lossG.backward()
        D_G_z2 = output.mean().item()               # Avg D score on fakes (after G update)
        optimizerG.step()                            # Update ONLY G

        # ─────────────────────────────────────────────────────────────────────
        # Logging
        G_losses.append(lossG.item())
        D_losses.append(lossD.item())

        if i % 100 == 0:
            print(f"{epoch+1:>5}/{NUM_EPOCHS} {i:>5}/{len(dataloader)} "
                  f"{lossD.item():>10.4f} {lossG.item():>10.4f} "
                  f"{D_x:>8.4f} {D_G_z1:.4f}/{D_G_z2:.4f}")

        # Save sample images for animation
        if (iters % 500 == 0) or (epoch == NUM_EPOCHS-1 and i == len(dataloader)-1):
            with torch.no_grad():
                fake_fixed = netG(fixed_noise).detach().cpu()
            img_list.append(vutils.make_grid(fake_fixed, padding=2, normalize=True))

        iters += 1

print("\n✅ Training complete!")

---
## Part 7: Visualise Training Progress

In [ ]:
# ── Plot Training Losses ──────────────────────────────────────────────────────
plt.figure(figsize=(12, 5))
plt.title("Generator and Discriminator Losses During Training", fontsize=14)
plt.plot(G_losses, label="Generator (G)",     alpha=0.7, color="steelblue")
plt.plot(D_losses, label="Discriminator (D)", alpha=0.7, color="tomato")
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("gan_losses.png", dpi=150)
plt.show()

print("""
📊 Interpreting the Loss Curves:
─────────────────────────────────────────────────────────────────
• D loss (~0.69 initially): log(0.5) + log(0.5) — both 50% guesses
• G loss drops as G improves at fooling D
• Healthy training: both losses oscillate without one collapsing
• Mode collapse symptom: G loss drops sharply, D loss spikes
─────────────────────────────────────────────────────────────────
""")

In [ ]:
# ── Real vs Fake Comparison ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Real images
axes[0].set_title("Real Images", fontsize=14)
axes[0].axis("off")
axes[0].imshow(
    np.transpose(
        vutils.make_grid(real_batch[0][:64].to(device), padding=5, normalize=True).cpu().numpy(),
        (1, 2, 0)
    ),
    cmap="gray"
)

# Generated images
axes[1].set_title("Generated Images (Epoch {})" .format(NUM_EPOCHS), fontsize=14)
axes[1].axis("off")
axes[1].imshow(
    np.transpose(img_list[-1].numpy(), (1, 2, 0)),
    cmap="gray"
)

plt.tight_layout()
plt.savefig("real_vs_fake.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Training Animation ────────────────────────────────────────────────────────
# Shows how G improves from random noise to coherent digits

fig = plt.figure(figsize=(8, 8))
plt.axis("off")
ims = [
    [plt.imshow(np.transpose(i.numpy(), (1, 2, 0)), animated=True, cmap="gray")]
    for i in img_list
]
ani = animation.ArtistAnimation(fig, ims, interval=500, repeat_delay=2000, blit=True)
plt.close()

# Display animation (works in Jupyter)
try:
    HTML(ani.to_jshtml())
except Exception:
    print("Animation display failed in this environment — view 'real_vs_fake.png' instead.")

---
## Part 8: Conceptual Questions (Reflection)

### ✏️ Exercise 8.1 — Short Answer Questions

Answer the following in your own words:

1. Why do we use `.detach()` when computing `lossD_fake` in Stage 1?
2. Why does the Generator's loss use `REAL_LABEL = 1.0` for fake images?
3. What would happen if you updated G and D **simultaneously** in a single backward pass?
4. Why is `LeakyReLU` preferred over `ReLU` in the Discriminator?
5. What is **mode collapse** and what does it look like in the generated images?
6. Why is `Tanh` used as the Generator's final activation instead of `Sigmoid`?
7. Why should we **not** use BatchNorm in the first Discriminator layer or the last Generator layer?

---

### ✅ Solutions: Conceptual Questions

**1. Why `.detach()`?**  
When training D, we feed it `fake = netG(noise)`. Without `.detach()`, PyTorch would build a computational graph all the way back through G. When we call `backward()`, gradients would flow through D **and** G, erroneously updating G's weights during Stage 1. `.detach()` cuts the graph — D's gradients stop at the fake image tensor and never reach G.

---

**2. Why `REAL_LABEL` for Generator's loss?**  
The Generator's goal is to **fool** D. A perfect G would produce images that D classifies as real (probability = 1). So we compute `BCE(D(G(z)), 1.0)` — the closer D's output is to 1, the lower G's loss. This is the non-saturating version of the GAN objective, which provides stronger gradients early in training compared to `log(1 - D(G(z)))`.

---

**3. Simultaneous updates?**  
If G and D were updated in one pass, their gradients would **interfere**. Specifically:
- When backpropagating through D for the real-vs-fake loss, gradients would also update G in a direction that tries to maximise D's loss — but D is also updating simultaneously. The minimax equilibrium breaks, and training becomes chaotic or diverges.

---

**4. LeakyReLU vs ReLU in Discriminator?**  
ReLU outputs 0 for all negative inputs, killing gradients for those neurons ("dead ReLU" problem). The Discriminator critically needs gradient flow so it can give the Generator useful learning signal. LeakyReLU with slope 0.2 allows a small gradient for negative activations, keeping neurons alive and training more stable.

---

**5. Mode Collapse?**  
Mode collapse occurs when G learns to produce only a **narrow variety** of outputs (e.g., only the digit "1" regardless of input noise) that consistently fool D. Visually, you see all generated images looking nearly identical. It happens because G finds a local optimum — a small set of images that D cannot distinguish from real — and stops exploring other modes of the data distribution.

---

**6. Tanh vs Sigmoid for Generator output?**  
The training data is normalised to `[-1, 1]` using `transforms.Normalize(0.5, 0.5)`. `Tanh` naturally outputs in `[-1, 1]`, so generated images live in the same space as real images. If we used `Sigmoid` (range `[0, 1]`), there would be a systematic mismatch: D would always see real images in `[-1, 1]` and fake images in `[0, 1]`, making the real-vs-fake distinction trivially easy and breaking training.

---

**7. BatchNorm placement?**  
- **First D layer**: The Discriminator's input is the raw image. BatchNorm at this layer normalises pixel statistics across the batch, potentially destroying the distribution difference between real and fake images that D is supposed to learn from.
- **Last G layer**: BatchNorm before `Tanh` would interfere with the final pixel values that need to be in `[-1, 1]`. The output distribution would be shifted/scaled by learnable BN parameters, making the range unpredictable.

---
## Part 9: Extension Exercises 🚀

If you finish early, try these extensions:

### Exercise 9.1 — Conditional GAN (cGAN)
Modify the Generator and Discriminator to accept a **class label** (digit 0–9) as input, so you can control which digit is generated.
> Hint: Use `nn.Embedding(num_classes, embed_dim)` and concatenate the embedding with the noise vector.

### Exercise 9.2 — Label Smoothing
Replace hard labels `(0, 1)` with soft labels like `(0.0–0.1, 0.9–1.0)` for the Discriminator.  
Does this stabilise training? Why might it help?

### Exercise 9.3 — Wasserstein GAN (WGAN)
Replace BCE loss with the Wasserstein distance approximation:
- Remove Sigmoid from D (make it a critic, not a classifier)
- D loss: `mean(D(fake)) - mean(D(real))` (maximise)
- G loss: `-mean(D(G(z)))` (minimise)
- Clip D weights to `[-0.01, 0.01]` after each update

### Exercise 9.4 — CIFAR-10
Switch the dataset to `torchvision.datasets.CIFAR10` (32×32 RGB).  
Change `NC=3` and adapt the architecture accordingly. Do you need more epochs? Larger `NGF`/`NDF`?

---

## Summary

| Concept | Key Takeaway |
|---|---|
| GAN Objective | Minimax game between G and D |
| Stage 1 (D) | D learns to tell real from fake; G weights **frozen** |
| Stage 2 (G) | G learns to fool D; D weights **frozen** |
| `.detach()` | Prevents gradient flow from D back through G during Stage 1 |
| ConvTranspose2d | Upsampling layer in Generator |
| Conv2d | Downsampling layer in Discriminator |
| LeakyReLU | Prevents dead neurons in Discriminator |
| BatchNorm | Stabilises training; skip on first D layer and last G layer |
| Tanh output | Matches data normalised to [-1, 1] |
| Mode collapse | G produces low-diversity outputs — a key failure mode |

---

*Lab designed for STEMvibe Deep Learning Series. Happy generating! 🎨*